# 💽 Lecture 5 (Part 2, Data Storage Types) – Data 100, Spring 2026

[Acknowledgments Page](https://ds100.org/sp26/acks/)

In [1]:
import numpy as np
import polars as pl

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
#%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 9)

sns.set()
sns.set_context('talk')
np.set_printoptions(threshold=20, precision=2, suppress=True)
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
# Use 2 decimal places for floats
pl.Config.set_float_precision(2)

# Silence some spurious seaborn warnings
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)


<br><br><br>

---

## 🤹‍♀️ File Formats Other than CSV

There are many file types for storing structured data: CSV, TSV, JSON, XML, ASCII, SAS...
* In lecture, we will cover JSON since `Polars` supports it out-of-box.

<br> <br>

---


### 🪆 JSON (JavaScript Object Notation)

The [congress.gov API](https://gpo.congress.gov/#/) (Application Programming Interface) provides data about the activities and members of the United States Congress (i.e., the House of Representatives and the Senate).

- Click the link above to see the kinds of information provided by the API. 

<br>

To get a JSON file containing information about the current members of Congress from California, you could use the following **API call**:

- `https://api.congress.gov/v3/member/CA?api_key=[INSERT_KEY]&limit=250&format=json&currentMember=True`

- You can instantly sign up for a congress.gov **API key** [here](https://gpo.congress.gov/sign-up/). Once you have your key, replace `[INSERT_KEY]` above with your key, and enter the API call as a URL in your browser. What happens? 

- Once the JSON from the API call is visible in your browser, you can click `File` --> `Save Page As` to save the JSON file to your coputer.

- Coarsely, API keys are used to track how much a given user engages with the API. There might be limits to the number of API calls (e.g., congress.gov API limits to 5,000 calls per hour), or a cost for API calls (e.g., using the OpenAI API for programmatically using ChatGPT).

<br>

For convenience, the JSON file from the call above has already been downloaded for you and is saved at `data/ca-congress-members.json`.

#### 📁 File contents

Let's look at `data/ca-congress-members.json` in the JupyterLab Explorer.

- Right-click the file, and then click `Open With` --> `Editor`.

- You'll notice that JSON looks a lot like a Python dictionary.

- Berkeley, CA is in the 12th district. Can you find our representative in Congress?

> Note: In general, it's a good idea to check the file size before opening a file in JupyterLab. Very large files can cause crashes. See `os.path.getsize` [documentation](https://docs.python.org/3/library/os.path.html#os.path.getsize).

We can programmatically view the first couple lines of the file using the same functions we used with CSVs:

In [3]:
congress_file = "data/ca-congress-members.json"

# Inspect the first five lines of the file
with open(congress_file, "r") as f:
    for i, row in enumerate(f):
        print(row)
        if i >= 4: break

{

    "members": [

        {

            "bioguideId": "T000491",

            "depiction": {



#### 🐍 EDA: Digging into JSON with Python

JSON data closely matches the internal Python object model.  

- In the following cell, we import the entire JSON datafile into a Python dictionary using the `json` package.

In [4]:
import json

# Import the JSON file into Python as a dictionary
with open(congress_file, "rb") as f:
    congress_json = json.load(f)

type(congress_json)

dict

The `congress_json` variable is a dictionary encoding the data in the JSON file.

Below, we access the first element of the `members` element of the `congress_json` dictionary.

- This first element is also a dictionary (and there are more dictionaries inside of it!)

In [5]:
# Grab the list corresponding to the `members` key in the JSON dictionary, 
# and then grab the first element of this list.
# In a moment, we'll see how we knew to use the key `members`, and that
# the resulting object is a list.
congress_json['members'][0]

{'bioguideId': 'T000491',
 'depiction': {'attribution': 'Image courtesy of the Member',
  'imageUrl': 'https://www.congress.gov/img/member/6774606d0b34857ecc9091a9_200.jpg'},
 'district': 45,
 'name': 'Tran, Derek',
 'partyName': 'Democratic',
 'state': 'California',
 'terms': {'item': [{'chamber': 'House of Representatives',
    'startYear': 2025}]},
 'updateDate': '2025-01-21T18:00:52Z',
 'url': 'https://api.congress.gov/v3/member/T000491?format=json'}

How should we probe a nested dictionary like `congress_json`?

We can start by identifying the top-level **keys** of the dictionary:

In [6]:
# Grab the top-level keys of the JSON dictionary
congress_json.keys()

dict_keys(['members', 'pagination', 'request'])

Looks like we have three top-level keys: `members`, `pagination`, and `request`.

> You'll often see a top-level `meta` key in JSON files. This does not refer to Meta (formerly Facebook). Instead, it typically refers to metadata (data about the data).  Metadata are often maintained alongside the data.

Let's check the type of the `members` element:

In [7]:
type(congress_json['members'])

list

Looks like a list! What are the first two elements?

In [8]:
congress_json['members'][:2]

[{'bioguideId': 'T000491',
  'depiction': {'attribution': 'Image courtesy of the Member',
   'imageUrl': 'https://www.congress.gov/img/member/6774606d0b34857ecc9091a9_200.jpg'},
  'district': 45,
  'name': 'Tran, Derek',
  'partyName': 'Democratic',
  'state': 'California',
  'terms': {'item': [{'chamber': 'House of Representatives',
     'startYear': 2025}]},
  'updateDate': '2025-01-21T18:00:52Z',
  'url': 'https://api.congress.gov/v3/member/T000491?format=json'},
 {'bioguideId': 'M001241',
  'depiction': {'attribution': 'Image courtesy of the Member',
   'imageUrl': 'https://www.congress.gov/img/member/67744ed90b34857ecc909155_200.jpg'},
  'district': 47,
  'name': 'Min, Dave',
  'partyName': 'Democratic',
  'state': 'California',
  'terms': {'item': [{'chamber': 'House of Representatives',
     'startYear': 2025}]},
  'updateDate': '2025-01-21T18:00:52Z',
  'url': 'https://api.congress.gov/v3/member/M001241?format=json'}]

More dictionaries! You can repeat the process above to traverse the nested dictionary.

You'll notice that each record of `congress_json['members']` looks like it could be a column of a DataFrame.

- The keys look a lot like column names, and the values could be the entries in each row.

<br> 

But, the two other elements of `congress_json` don't have the same structure as `congress_json['members']`.

- So, they probably don't belong in a DataFrame containing the members of Congress from CA.

- We'll see the implications of this inconsistency in the next section.

In [9]:
print(congress_json['pagination'])
print(congress_json['request'])

{'count': 54}
{'contentType': 'application/json', 'format': 'json'}


#### 🐻‍❄️ JSON with Polars

`Polars` has a built in function called `pl.read_json` for reading in JSON files.

- Uncomment the code below and see what happens.

In [10]:
# pl.read_json(congress_file)

Uh oh. That gives us a single row with three columns, each one holding a whole nested piece of the file.

- The code above imports the entire JSON file located at `congress_file` (`congress_json`), including `congress_json['pagination']` and `congress_json['request']`.

- We only want to make a DataFrame out of `congress_json['members']`, with one row per member.

This time, let's try converting the `members` element of `congress_json` to a DataFrame by using `pl.DataFrame`:

In [11]:
# Convert dictionary to DataFrame
congress_df = pl.DataFrame(congress_json['members'])
congress_df.head()

bioguideId,depiction,district,name,partyName,state,terms,updateDate,url
str,struct[2],i64,str,str,str,struct[1],str,str
"""T000491""","{""Image courtesy of the Member"",""https://www.congress.gov/img/member/6774606d0b34857ecc9091a9_200.jpg""}",45,"""Tran, Derek""","""Democratic""","""California""","{[{""House of Representatives"",null,2025}]}","""2025-01-21T18:00:52Z""","""https://api.congress.gov/v3/me…"
"""M001241""","{""Image courtesy of the Member"",""https://www.congress.gov/img/member/67744ed90b34857ecc909155_200.jpg""}",47,"""Min, Dave""","""Democratic""","""California""","{[{""House of Representatives"",null,2025}]}","""2025-01-21T18:00:52Z""","""https://api.congress.gov/v3/me…"
"""K000400""","{""Image courtesy of the Member"",""https://www.congress.gov/img/member/k000400_200.jpg""}",37,"""Kamlager-Dove, Sydney""","""Democratic""","""California""","{[{""House of Representatives"",null,2023}]}","""2025-01-21T18:00:52Z""","""https://api.congress.gov/v3/me…"
"""G000598""","{""Image courtesy of the Member"",""https://www.congress.gov/img/member/g000598_200.jpg""}",42,"""Garcia, Robert""","""Democratic""","""California""","{[{""House of Representatives"",null,2023}]}","""2025-01-21T18:00:52Z""","""https://api.congress.gov/v3/me…"
"""K000397""","{""Image courtesy of the Member"",""https://www.congress.gov/img/member/k000397_200.jpg""}",40,"""Kim, Young""","""Republican""","""California""","{[{""House of Representatives"",null,2021}]}","""2025-01-21T18:00:52Z""","""https://api.congress.gov/v3/me…"


We've successfully begun to rectangularize our JSON data!

<br><br><br>

**Instructor Note: Return to Slides!**

<br/>

---


## 🕰️ Temporality

Let's briefly look at how we can use the `Polars` `dt` accessors to work with dates/times in a dataset.

We will use the Berkeley Police Department (PD) Calls for Service dataset.

In [12]:
calls = pl.read_csv("data/Berkeley_PD_-_Calls_for_Service.csv")
calls.head()

CASENO,OFFENSE,EVENTDT,EVENTTM,CVLEGEND,CVDOW,InDbDate,Block_Location,BLKADDR,City,State
i64,str,str,str,str,i64,str,str,str,str,str
21014296,"""THEFT MISD. (UNDER $950)""","""04/01/2021 12:00:00 AM""","""10:58""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21014391,"""THEFT MISD. (UNDER $950)""","""04/01/2021 12:00:00 AM""","""10:38""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21090494,"""THEFT MISD. (UNDER $950)""","""04/19/2021 12:00:00 AM""","""12:15""","""LARCENY""",1,"""06/15/2021 12:00:00 AM""","""2100 BLOCK HASTE ST Berkeley, …","""2100 BLOCK HASTE ST""","""Berkeley""","""CA"""
21090204,"""THEFT FELONY (OVER $950)""","""02/13/2021 12:00:00 AM""","""17:00""","""LARCENY""",6,"""06/15/2021 12:00:00 AM""","""2600 BLOCK WARRING ST Berkeley…","""2600 BLOCK WARRING ST""","""Berkeley""","""CA"""
21090179,"""BURGLARY AUTO""","""02/08/2021 12:00:00 AM""","""6:20""","""BURGLARY - VEHICLE""",1,"""06/15/2021 12:00:00 AM""","""2700 BLOCK GARBER ST Berkeley,…","""2700 BLOCK GARBER ST""","""Berkeley""","""CA"""


Looks like there are three columns with dates/times: `EVENTDT`, `EVENTTM`, and `InDbDate`. 

- `EVENTDT` stands for the **date** when the event took place

- `EVENTTM` stands for the **time of day** the event took place (in 24-hr format)

- `InDbDate` is the date this call is entered into the database.

We can convert these string columns to `datetime` objects using the `str.to_datetime` method.

In [13]:
# str.to_datetime() will try to work out the layout of the string on its own,
# and it gives up when the layout is ambiguous or unusual.
# It's good practice to specify the format of your datetimes.
# See the documentation and the `format` argument.

calls = calls.with_columns(
    pl.col("EVENTDT").str.to_datetime(format='%m/%d/%Y %I:%M:%S %p')
)

calls.head()

CASENO,OFFENSE,EVENTDT,EVENTTM,CVLEGEND,CVDOW,InDbDate,Block_Location,BLKADDR,City,State
i64,str,datetime[μs],str,str,i64,str,str,str,str,str
21014296,"""THEFT MISD. (UNDER $950)""",2021-04-01 00:00:00,"""10:58""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21014391,"""THEFT MISD. (UNDER $950)""",2021-04-01 00:00:00,"""10:38""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21090494,"""THEFT MISD. (UNDER $950)""",2021-04-19 00:00:00,"""12:15""","""LARCENY""",1,"""06/15/2021 12:00:00 AM""","""2100 BLOCK HASTE ST Berkeley, …","""2100 BLOCK HASTE ST""","""Berkeley""","""CA"""
21090204,"""THEFT FELONY (OVER $950)""",2021-02-13 00:00:00,"""17:00""","""LARCENY""",6,"""06/15/2021 12:00:00 AM""","""2600 BLOCK WARRING ST Berkeley…","""2600 BLOCK WARRING ST""","""Berkeley""","""CA"""
21090179,"""BURGLARY AUTO""",2021-02-08 00:00:00,"""6:20""","""BURGLARY - VEHICLE""",1,"""06/15/2021 12:00:00 AM""","""2700 BLOCK GARBER ST Berkeley,…","""2700 BLOCK GARBER ST""","""Berkeley""","""CA"""


Now we can use the `dt` accessor on this column.

We can get the month:

In [14]:
# 1 - January, 2 - February, ..., 12 - December
calls["EVENTDT"].dt.month()

EVENTDT
i8
4
4
4
2
2
12
5
3
3


Which day of the week the date is on:

In [15]:
# 1 - Monday, 2 - Tuesday, ..., 7 - Sunday
calls["EVENTDT"].dt.weekday()

EVENTDT
i8
4
4
1
6
1
6
1
7
3


We can also sort by datetime:

In [16]:
# Sort the DataFrame by datetime to find the earliest call.
calls.sort("EVENTDT").head()

CASENO,OFFENSE,EVENTDT,EVENTTM,CVLEGEND,CVDOW,InDbDate,Block_Location,BLKADDR,City,State
i64,str,datetime[μs],str,str,i64,str,str,str,str,str
20092214,"""THEFT FROM AUTO""",2020-12-17 00:00:00,"""18:30""","""LARCENY - FROM VEHICLE""",4,"""06/15/2021 12:00:00 AM""","""800 BLOCK SHATTUCK AVE Berkele…","""800 BLOCK SHATTUCK AVE""","""Berkeley""","""CA"""
20057373,"""GUN/WEAPON""",2020-12-17 00:00:00,"""22:18""","""WEAPONS OFFENSE""",4,"""06/15/2021 12:00:00 AM""","""6200 BLOCK SAN PABLO AVE Berke…","""6200 BLOCK SAN PABLO AVE""","""Berkeley""","""CA"""
20057207,"""ASSAULT/BATTERY MISD.""",2020-12-17 00:00:00,"""16:50""","""ASSAULT""",4,"""06/15/2021 12:00:00 AM""","""2100 BLOCK SHATTUCK AVE Berkel…","""2100 BLOCK SHATTUCK AVE""","""Berkeley""","""CA"""
20057324,"""THEFT MISD. (UNDER $950)""",2020-12-17 00:00:00,"""15:44""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""1800 BLOCK 4TH ST Berkeley, CA…","""1800 BLOCK 4TH ST""","""Berkeley""","""CA"""
20057573,"""BURGLARY RESIDENTIAL""",2020-12-17 00:00:00,"""22:15""","""BURGLARY - RESIDENTIAL""",4,"""06/15/2021 12:00:00 AM""","""1700 BLOCK STUART ST Berkeley,…","""1700 BLOCK STUART ST""","""Berkeley""","""CA"""


We can also do many things with the `dt` accessor like switching time zones and converting time back to UNIX/POSIX time. Check out the documentation on the [`.dt` accessor](https://docs.pola.rs/api/python/stable/reference/series/temporal.html) and [time series/date functionality](https://docs.pola.rs/user-guide/transformations/time-series/parsing/).

What type are datetime objects?

In [17]:
calls["EVENTDT"].dtype

Datetime(time_unit='us', time_zone=None)

`us` above stands for microseconds, the time unit `Polars` records by default.

- The other choices of time unit are milliseconds (`ms`) and nanoseconds (`ns`).

Under the hood, datetimes are integers representing the number of **microseconds** since 1/1/1970 UTC.

In [18]:
# datetimes are stored as integers representing number of
# microseconds since 1970-01-01
calls["EVENTDT"].cast(pl.Int64)

EVENTDT
i64
1617235200000000
1617235200000000
1618790400000000
1613174400000000
1612742400000000
1608940800000000
1620000000000000
1615075200000000
1617148800000000


<br><br><br>

**Instructor Note: Return to Slides!**

<br/>

---


## 🤷 Faithfulness and missing values

To conclude, let's **very** briefly explore missingness in the Berkeley PD Calls for Service dataset.

Looking at the top of the dataframe, we can already see that there are missing values in the `BLKADDR` column.

In [19]:
calls.head()

CASENO,OFFENSE,EVENTDT,EVENTTM,CVLEGEND,CVDOW,InDbDate,Block_Location,BLKADDR,City,State
i64,str,datetime[μs],str,str,i64,str,str,str,str,str
21014296,"""THEFT MISD. (UNDER $950)""",2021-04-01 00:00:00,"""10:58""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21014391,"""THEFT MISD. (UNDER $950)""",2021-04-01 00:00:00,"""10:38""","""LARCENY""",4,"""06/15/2021 12:00:00 AM""","""Berkeley, CA (37.869058, -122.…",null,"""Berkeley""","""CA"""
21090494,"""THEFT MISD. (UNDER $950)""",2021-04-19 00:00:00,"""12:15""","""LARCENY""",1,"""06/15/2021 12:00:00 AM""","""2100 BLOCK HASTE ST Berkeley, …","""2100 BLOCK HASTE ST""","""Berkeley""","""CA"""
21090204,"""THEFT FELONY (OVER $950)""",2021-02-13 00:00:00,"""17:00""","""LARCENY""",6,"""06/15/2021 12:00:00 AM""","""2600 BLOCK WARRING ST Berkeley…","""2600 BLOCK WARRING ST""","""Berkeley""","""CA"""
21090179,"""BURGLARY AUTO""",2021-02-08 00:00:00,"""6:20""","""BURGLARY - VEHICLE""",1,"""06/15/2021 12:00:00 AM""","""2700 BLOCK GARBER ST Berkeley,…","""2700 BLOCK GARBER ST""","""Berkeley""","""CA"""


We can use the `.is_null()` method to get a sense of how often values in `BLKADDR` are missing.

- `null` is how `Polars` records a missing entry.

In [20]:
# is_null() returns a Series of booleans indicating whether each element
# # in the Series is missing.
print(calls['BLKADDR'].is_null().head())

# The mean of a Series of booleans is the proportion of booleans that are True.
calls['BLKADDR'].is_null().mean()

shape: (10,)
Series: 'BLKADDR' [bool]
[
	true
	true
	false
	false
	false
	false
	false
	false
	false
	false
]


0.007598784194528876

It looks like missing values are actually quite rare: Only 0.8% of records are missing a value in `BLKADDR`.

Why are these values missing? 

- Again, looking at just the first few rows, we see that `null` values in `BLKADDR` appear to be accompanied by latitude/longitude coordinates in the `Block_Location` column.

- In all likelihood, missing values in `BLKADDR` probably correspond to locations that do not have a defined address in the officer's navigation or GPS system.

<br>

The best default approach here: Leave the rows with missing `BLKADDR` untouched, or replace the `null` values with a `MISSING` indicator.

- In the future, if we wanted to conduct an analysis of the streets where police incidents were most common, we might impute `BLKADDR` by using the nearest street, which we could identify with an external package.

<br>

For a very rough sense of missingness in each column of a DataFrame, you can use the `null_count()` method.

- Based on the output, it looks like the only column with missing values is `BLKADDR`.

In [21]:
# null_count() reports the number of missing entries in each column.
# Compare the count for BLKADDR to the total number of rows.
print(calls.height, "rows")
calls.null_count()

2632 rows


CASENO,OFFENSE,EVENTDT,EVENTTM,CVLEGEND,CVDOW,InDbDate,Block_Location,BLKADDR,City,State
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,20,0,0


<br><br><br>

**Instructor Note: Return to Slides!**